## 1. ¿Qué es Gradio?
Gradio es una biblioteca de Python que permite crear interfaces web personalizables de forma rápida y sencilla para modelos de Machine Learning. Con solo unas pocas líneas de código podemos construir una interfaz con entradas (texto, imágenes, audio, etc.) y salidas (predicciones, gráficos, texto, etc.), y compartirla fácilmente mediante un enlace público o desplegarla en la nube.

Características principales:
- Sencillez: Define la función que procesa los datos y Gradio construye la interfaz automáticamente.

- Componentes variados: Entradas y salidas como Textbox, Slider, Image, Audio, DataFrame, etc.

- Compatibilidad con modelos: Funciona con cualquier modelo de Python (scikit-learn, PyTorch, TensorFlow, transformers, etc.).

- Despliegue permanente: Puedes subir tu aplicación a Hugging Face Spaces, una plataforma gratuita de alojamiento.



In [ ]:
# Instalación de Gradio
%pip install --upgrade gradio

## 2. Crear interfaces con Gradio (vistazo rápido)
Gradio ofrece dos formas principales de construir interfaces:

Interfaz simple (gr.Interface): Ideal para casos de una sola entrada y una salida. Se define con fn, inputs, outputs y title.

Interfaz con más control (gr.Blocks): Permite diseñar layouts complejos, con pestañas, filas, columnas y múltiples componentes interactivos.

Ejemplo pequeño con gr.Interface:

In [1]:
import gradio as gr

def saludar(nombre):
    return f"¡Hola {nombre}!"

demo = gr.Interface(fn=saludar, inputs="text", outputs="text")
demo.launch()

/usr/local/anaconda3/envs/classes/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio/flagged/dataset1.csv


Con gr.Blocks podemos personalizar más:

In [2]:
with gr.Blocks() as demo:
    gr.Markdown("# Mi Aplicación")
    nombre = gr.Textbox(label="Nombre")
    salida = gr.Textbox(label="Saludo")
    boton = gr.Button("Saludar")
    boton.click(fn=saludar, inputs=nombre, outputs=salida)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


3. Despliegue en Hugging Face Spaces
Hugging Face Spaces es un servicio gratuito que permite alojar aplicaciones de Machine Learning, especialmente aquellas construidas con Gradio, Streamlit o Docker. Para desplegar nuestra aplicación necesitamos:

1. Crear una cuenta en huggingface.co.

2. Crear un nuevo Space:

- Ve a tu perfil y haz clic en "New Space".

- Elige un nombre para tu Space.

- Selecciona "Gradio" como SDK.

- Puedes elegir visibilidad pública o privada (gratuita para público).

- Clic en "Create Space".

3. Subir los archivos:

- Tu aplicación debe tener un archivo app.py (punto de entrada) y un archivo requirements.txt con las dependencias.
- Puedes subir los archivos mediante la interfaz web (arrastrando) o usando Git.

La aplicación funciona si tu en tu space ves "Running..."



## Ejemplo de chatbot con Gradio

In [ ]:
import gradio as gr
import rag_engine  # importamos el rag_engine con la logica del chatbot)

def ask(query: str, top_k: int, umbral: float):
    """
    Función principal que llamará Gradio
    Entrada:
    - query: pregunta del usuarii
    - top_k: top_k documentos relevantes seleccionados
    - umbral: umbral de similitud

    Salida:
    - respuesta: respuesta generada
    - docs_text: documentos recuperados
    """

    # Devuelve la respuesta generada
    docs = rag_engine.recuperar_documentos(query, int(top_k), float(umbral))
    answer = rag_engine.generar_respuesta(query, docs)

    # Formatea los documentos para mostrarlos en Gradio
    if docs:
        docs_text = "\n\n---\n\n".join(docs)
    else:
        docs_text = "No se encontraron documentos relevantes con el umbral especificado."

    return answer, docs_text

# Creamos la interfaz de Gradio
with gr.Blocks(title="Chat RAG", theme=gr.themes.Soft()) as demo:
    # Texto
    gr.Markdown("# Sistema de preguntas con RAG")
    gr.Markdown("Haz una pregunta sobre la base de conocimiento (documents.json).")

    with gr.Row():
        with gr.Column(scale=4):
            # Campo de input
            query_input = gr.Textbox(
                label="Tu pregunta (en inglés)",
                placeholder="Ej: Where is the hospital?"
            )
        with gr.Column(scale=1):
            # Campos de input con sliders para top_k y umbral
            top_k_input = gr.Slider(
                1, 5, value=1, step=1,
                label="Top‑k documentos"
            )
            umbral_input = gr.Slider(
                0.0, 1.0, value=0.55, step=0.05,
                label="Umbral de similitud"
            )

    # Campos de output
    answer_output = gr.Textbox(label="Respuesta", lines=3)
    docs_output = gr.Textbox(label="Documentos recuperados", lines=6, max_lines=15)

    # Botón de envio
    submit_btn = gr.Button("Enviar")
    submit_btn.click(
        fn=ask,
        inputs=[query_input, top_k_input, umbral_input],
        outputs=[answer_output, docs_output]
    )

# Lanzamos la interfaz de Gradio
demo.launch()

## 5. Llamar a la interfaz como API

Una de las grandes ventajas de Gradio es que cualquier interfaz puede ser utilizada como API REST de forma automática. Esto es especialmente útil si queremos integrar nuestra aplicación en otro sistema o hacer consultas programáticas.

### Opción 1: Cliente oficial de Gradio (gradio_client)
Gradio proporciona una biblioteca cliente que permite conectarse a cualquier espacio de Hugging Face o a una aplicación Gradio local.

In [ ]:
# instalación de gradio_client
%pip install gradio_client

Haz un test de tu api en un nuevo archivo .py con lo siguiente:

In [ ]:
# Una vez hemos hecho el deploy en Hugging Face, podemos probar la API de Gradio
from gradio_client import Client

client = Client("nombre-usuario/nombre-space")
result = client.predict(
	query="Where is the hospital located!!",
	top_k=5,
	umbral=0.55,
	api_name="/ask"
)
print(result[0])